# EDGAR

**Research Question:** How does stress in the non-financial commercial paper market influence firms' working capital management and supply-chain operations?

**Scope:** Firm identification via the SEC EDGAR full-text search API (`efts.sec.gov`). Non-financial firms that disclosed an active commercial paper program in their annual 10-K filings are identified for fiscal years 2007 (pre-GFC) and 2019 (pre-COVID-19). The resulting sample forms the basis for subsequent firm-level analysis of working capital management and supply-chain behavior.

**Episode selection:** The 9/11 episode is excluded on two grounds: EDGAR full-text search coverage begins in 2001, precluding pre-event 10-K identification; and the CSI confirms that the stress episode did not sustain above the 90th percentile threshold for two or more consecutive weeks, making it unsuitable for the firm-level stress window analysis.

**Output:** A deduplicated list of CIK numbers, company names, and SIC codes saved to `data/ED_cp_final_sample.csv`, ready for matching to Compustat or Capital IQ for financial data.

## Section 1 – Imports & Configuration

In [4]:
# Imports
import warnings
from urllib3.exceptions import NotOpenSSLWarning
warnings.filterwarnings("ignore", category=NotOpenSSLWarning)

# Core Data & API
import requests
import pandas as pd
import re

# File system & config
from dotenv import load_dotenv
import os
import sys
from pathlib import Path

# Time handling
import time

# Configuration
sys.path.insert(0, str(Path().resolve().parent))
from config import *

load_dotenv(ANALYSIS_DIR / ".env")
HEADERS = {"User-Agent": os.getenv("USER_AGENT", "MyBot/1.0 (contact@example.com)")}

EDGAR_BASE_URL = "https://efts.sec.gov/LATEST/search-index"

## Section 2 – EDGAR Search

10-K filings are searched for fiscal years 2007 and 2019 (pre-GFC and pre-COVID-19). Filings containing the phrases `"commercial paper program"` or `"commercial paper facility"` are retrieved and parsed into a flat record structure for downstream filtering.

In [5]:
# CELL 2.1 — EDGAR Search Helper Functions

def search_edgar(query, year, form="10-K"):
    """Paginate through EDGAR full-text search results for a given query and filing year."""
    hits  = []
    start = 0

    while True:
        params = {
            "q":         f'"{query}"',
            "forms":     form,
            "dateRange": "custom",
            "startdt":   f"{year}-01-01",
            "enddt":     f"{year}-12-31",
            "from":      start,
        }

        response = requests.get(EDGAR_BASE_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        batch = data.get("hits", {}).get("hits", [])
        if not batch:
            break

        hits.extend(batch)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        print(f"  '{query}' {year}: fetched {len(hits)}/{total}", end="\r")

        if len(hits) >= total:
            break

        start += 10
        time.sleep(0.15)

    print()
    return hits


def parse_hits(hits, year, query):
    """Extract relevant fields from raw EDGAR search hits into a flat record list."""
    records = []
    for h in hits:
        src = h.get("_source", {})
        records.append({
            "cik":          (src.get("ciks")          or [""])[0],
            "company_name": (src.get("display_names") or [""])[0],
            "sic":          (src.get("sics")          or [""])[0],
            "filing_date":  src.get("file_date", ""),
            "accession_no": src.get("adsh", ""),
            "search_year":  year,
            "query":        query,
        })
    return records

In [6]:
# CELL 2.2 — Run Search

QUERIES = ["commercial paper program", "commercial paper facility"]
YEARS   = [2007, 2019]

all_records = []

for year in YEARS:
    for query in QUERIES:
        print(f"Searching: '{query}' | year: {year}")
        hits    = search_edgar(query, year)
        records = parse_hits(hits, year, query)
        all_records.extend(records)
        time.sleep(0.5)

df = pd.DataFrame(all_records)
print(f"\nTotal raw records: {len(df)}")

Searching: 'commercial paper program' | year: 2007
  'commercial paper program' 2007: fetched 400/329
Searching: 'commercial paper facility' | year: 2007
  'commercial paper facility' 2007: fetched 46/46
Searching: 'commercial paper program' | year: 2019
  'commercial paper program' 2019: fetched 300/253
Searching: 'commercial paper facility' | year: 2019
  'commercial paper facility' 2019: fetched 21/21

Total raw records: 767


### 2.3 Filtering

Financial firms (SIC 6000–6999), utilities (SIC 4900–4999), pipelines and MLPs (SIC 4610–4619), equipment leasing (SIC 7350–7359), petroleum wholesalers (SIC 5170–5172), vehicle rental (SIC 7510–7519), and public administration (SIC 9000–9999) are excluded. These sectors are dropped because rate regulation, asset-backed structures, or non-operating mandates make their liquidity management incomparable to industrial CP issuers. Within the remaining firms, duplicates are resolved by keeping the earliest filing per firm per search year.

In [7]:
# CELL 2.3 — Filter and Deduplicate

df["sic"] = pd.to_numeric(df["sic"], errors="coerce")
nan_sic   = df[df["sic"].isna()][["company_name", "search_year"]].drop_duplicates()
print(f"Firms with unparseable SIC (will be dropped): {len(nan_sic)} \n")
print(nan_sic.to_string())

df_nonfinancial = df[
    df["sic"].notna()          &
    ~df["sic"].between(6000, 6999) &   # financials
    ~df["sic"].between(4900, 4999) &   # utilities
    ~df["sic"].between(4610, 4619) &   # pipelines / MLPs
    ~df["sic"].between(9000, 9999) &   # public administration
    ~df["sic"].between(7350, 7359) &   # equipment / vehicle leasing
    ~df["sic"].between(5170, 5172) &   # petroleum / pipeline wholesalers
    ~df["sic"].between(7510, 7519)     # vehicle rental
].copy()

df_nonfinancial["sic"] = df_nonfinancial["sic"].astype(int)

# Deduplicate: one row per firm per search year, keep earliest filing
df_deduped = (
    df_nonfinancial
    .sort_values("filing_date")
    .drop_duplicates(subset=["cik", "search_year"])
    .reset_index(drop=True)
)

print(f"\nAfter removing financials and duplicates: {len(df_deduped)} firm-year observations")
print(f"Unique firms: {df_deduped['cik'].nunique()}")

Firms with unparseable SIC (will be dropped): 2 

                                       company_name  search_year
96     ALLSTATE LIFE INSURANCE CO  (CIK 0000352736)         2007
197  NATIONWIDE LIFE INSURANCE CO  (CIK 0000205695)         2007

After removing financials and duplicates: 159 firm-year observations
Unique firms: 138


## Section 3 – Validation

Each filing is fetched from the SEC EDGAR archives and scanned for ownership language around `"commercial paper"` mentions — phrases such as *"our commercial paper program"* or *"we may issue commercial paper"* confirm that the firm is an issuer, not merely referencing a counterparty's program. Filings that cannot be fetched or lack ownership language are flagged for manual review.

In [8]:
# CELL 3.1 — Validation Helper Functions

OWNERSHIP = re.compile(
    r"\b(our|we|the company|company'?s|registrant|issuer|we maintain|we have|we may issue)\b",
    re.IGNORECASE
)


def get_filing_doc_url(accession_no, cik):
    """
    Use the EDGAR filing index JSON to find the primary 10-K document
    by selecting the largest .htm file in the filing directory.
    """
    acc_clean = accession_no.replace("-", "")
    cik_clean = cik.lstrip("0")
    index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_clean}/{acc_clean}/index.json"

    try:
        r = requests.get(index_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        items = r.json().get("directory", {}).get("item", [])

        # Pick the largest .htm file — almost always the main 10-K document
        htm_files = [
            item for item in items
            if item["name"].endswith(".htm")
            and not item["name"].startswith("0000")  # exclude index files
            and item["size"] != ""
        ]

        if htm_files:
            largest = max(htm_files, key=lambda x: int(x["size"]))
            return f"https://www.sec.gov/Archives/edgar/data/{cik_clean}/{acc_clean}/{largest['name']}"

    except Exception:
        return None


def fetch_filing_text(url, max_chars=3_000_000):
    """
    Fetch raw filing text, truncated to avoid excessive memory use.
    The 3M character cap is conservative — CP disclosures appear well within this limit.
    """
    if not url:
        return None
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        return r.text[:max_chars]
    except Exception:
        return None


def extract_cp_contexts(text, window=400):
    """Return all text snippets around 'commercial paper' mentions."""
    if not text:
        return []
    contexts = []
    for m in re.finditer(r"commercial paper", text, re.IGNORECASE):
        start = max(0, m.start() - window)
        end   = min(len(text), m.end() + window)
        contexts.append(text[start:end])
    return contexts


def classify_filing(contexts):
    """
    Returns:
        'approved'     – at least one context contains ownership language
        'needs_review' – no ownership language found
        'no_text'      – filing could not be fetched
    """
    if not contexts:
        return "no_text"
    if any(OWNERSHIP.search(c) for c in contexts):
        return "approved"
    return "needs_review"

In [9]:
# CELL 3.2 — Run Validation

results = []

for i, row in df_deduped.iterrows():
    url      = get_filing_doc_url(row["accession_no"], row["cik"])
    text     = fetch_filing_text(url)
    contexts = extract_cp_contexts(text)
    status   = classify_filing(contexts)

    results.append({
        "cik":          row["cik"],
        "company_name": row["company_name"],
        "sic":          row["sic"],
        "search_year":  row["search_year"],
        "accession_no": row["accession_no"],
        "status":       status,
        "n_mentions":   len(contexts),
        "url":          url or "",
    })

    print(f"[{i+1:>3}/{len(df_deduped)}] {status:>12}  |  {row['company_name'][:50]}", end="\r")
    time.sleep(0.6)

print("\nDone.")

df_results = pd.DataFrame(results)
print(df_results["status"].value_counts())

df_review   = df_results[df_results["status"] == "needs_review"]
df_approved = df_results[df_results["status"] == "approved"]

[159/159]     approved  |  CABOT CORP  (CBT)  (CIK 0000016040) 0001140859)000
Done.
status
approved        151
needs_review      5
no_text           3
Name: count, dtype: int64


## Section 4 – Final Sample Construction

Manual overrides are applied to the validation results. Firms flagged as `needs_review` or `no_text` were inspected individually; confirmed issuers are added via `MANUAL_INCLUDE` and non-issuers or structurally incomparable firms via `MANUAL_EXCLUDE` with reasons noted inline. CIK numbers are normalized for WRDS matching and firms appearing in both crisis periods are flagged.

In [11]:
MANUAL_INCLUDE = [
    "0000012355",  # Black & Decker Corp         – confirmed issuer
    "0000055785",  # Kimberly-Clark Corp         – confirmed issuer
    "0000045876",  # Harsco Corp                 – confirmed issuer
    "0000916076",  # Martin Marietta Materials   – confirmed issuer
    "0001090012",  # Devon Energy Corp           – confirmed issuer
    "0000764180",  # Altria Group                – confirmed issuer
]

MANUAL_EXCLUDE = [
    "0001043277",  # C H Robinson Worldwide      – no active CP program found
    "0001449488",  # CSI Compressco LP           – no active CP program found
    "0001173911",  # Enbridge Energy Mgmt LLC    – non-operating LP manager
    "0001657788",  # Kimbell Royalty Partners    – pure royalty vehicle, no operations
    "0001403161",  # Visa Inc.                   – payment network / financial intermediary
    "0001141391",  # Mastercard Inc.             – payment network / financial intermediary
    "0001365135",  # Western Union               – payment services / financial intermediary
    "0001136893",  # Fidelity National Info Svcs – fintech / payment services
]

df_manual_include = df_results[df_results["cik"].isin(MANUAL_INCLUDE)].copy()
df_manual_include["status"] = "manual_approved"

df_final = (
    pd.concat([df_approved, df_manual_include], ignore_index=True)
    .pipe(lambda df: df[~df["cik"].isin(MANUAL_EXCLUDE)])
    .drop_duplicates(subset=["cik", "search_year"])
    .reset_index(drop=True)
)

# Normalize CIK for WRDS matching
df_final["cik_norm"] = df_final["cik"].astype(str).str.lstrip("0")

# Flag firms appearing in both crisis periods
cp_counts = df_final.groupby("cik_norm")["search_year"].nunique()
df_final["in_both_periods"] = df_final["cik_norm"].map(cp_counts > 1)

# Keep only what is needed downstream
df_final = df_final[["cik_norm", "search_year", "sic", "in_both_periods", "status", "accession_no", "company_name"]]

print(f"Final sample: {len(df_final)} firm-year observations")
print(f"Unique firms: {df_final['cik_norm'].nunique()}")
print(f"In both periods: {df_final['in_both_periods'].sum() // 2} firms")
print(f"\nStatus breakdown:")
print(df_final["status"].value_counts())

df_final.to_csv(DATA_DIR / "ED_cp_final_sample.csv", index=False)
print(f"\nSaved: ED_cp_final_sample.csv")

Final sample: 150 firm-year observations
Unique firms: 130
In both periods: 20 firms

Status breakdown:
status
approved           144
manual_approved      6
Name: count, dtype: int64

Saved: ED_cp_final_sample.csv
